# Awinda outlier check: 3-DoF vs 1-DoF ID

Flag channel-level Awinda ID predictions where **RMSE > 0.25 N·m/kg** or **R² < 0.6**, for:

- **3-DoF / multi-joint**: `process_awinda.npz`
- **1-DoF / single-joint**: `process_awinda_per_joint_{hip,knee,ankle}.npz`

Nothing is dropped from the caches — this notebook is for inspection only.

Interactive viewer plots **OpenSim ID (GT)** vs **model** moments (same style as `visualize_awinda*.ipynb`).

Run `process_awinda.ipynb` and `process_awinda_per_joint.ipynb` first if caches are missing.


In [1]:
import io
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
FULL_CACHE = CACHE_DIR / 'process_awinda.npz'
JOINT_CACHE_PATHS = {
    'hip': CACHE_DIR / 'process_awinda_per_joint_hip.npz',
    'knee': CACHE_DIR / 'process_awinda_per_joint_knee.npz',
    'ankle': CACHE_DIR / 'process_awinda_per_joint_ankle.npz',
}

# Channel-level outlier thresholds (as requested)
MAX_RMSE = 0.25   # N·m/kg
MIN_R2 = 0.6

FULL_DISPLAY_NAMES = {
    'hip_flexion_r': 'Hip R', 'knee_angle_r': 'Knee R', 'ankle_angle_r': 'Ankle R',
    'hip_flexion_l': 'Hip L', 'knee_angle_l': 'Knee L', 'ankle_angle_l': 'Ankle L',
}
JOINT_DISPLAY = {
    'hip': ['Hip R', 'Hip L'],
    'knee': ['Knee R', 'Knee L'],
    'ankle': ['Ankle R', 'Ankle L'],
}

print('Full cache:', FULL_CACHE, '| exists=', FULL_CACHE.is_file())
for j, p in JOINT_CACHE_PATHS.items():
    print(f'  {j:5s}', p.name, '| exists=', p.is_file())
print(f'Flag if RMSE > {MAX_RMSE} N·m/kg or R² < {MIN_R2}')


Full cache: /home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda.npz | exists= True
  hip   process_awinda_per_joint_hip.npz | exists= True
  knee  process_awinda_per_joint_knee.npz | exists= True
  ankle process_awinda_per_joint_ankle.npz | exists= True
Flag if RMSE > 0.25 N·m/kg or R² < 0.6


In [2]:
def _trial_key_to_prefix(trial_key: str) -> str:
    return trial_key.replace('::', '__')


def _load_trial_map(data, channels: List[str], *, joint: Optional[str] = None) -> Dict[str, dict]:
    sync_method = str(data['sync_method'].item()) if 'sync_method' in data.files else 'angle_xcorr'
    trial_data = {}
    for trial_key in data['trial_keys']:
        key = str(trial_key)
        p = _trial_key_to_prefix(key)
        meta = data[f'{p}__meta']
        rmse = np.asarray(data[f'{p}__rmse_nmpkg'], dtype=float)
        r2 = np.asarray(data[f'{p}__r2_nmpkg'], dtype=float)
        metrics = [
            {'channel': channels[c], 'rmse_nmpkg': float(rmse[c]), 'r2_nmpkg': float(r2[c])}
            for c in range(len(channels))
        ]
        entry = {
            'subject': str(meta[0]),
            'condition': str(meta[1]),
            'mass_kg': float(meta[2]),
            't': np.asarray(data[f'{p}__t'], dtype=np.float64),
            'pred_nmpkg': np.asarray(data[f'{p}__pred_nmpkg'], dtype=np.float32),
            'id_nmpkg': np.asarray(data[f'{p}__id_nmpkg'], dtype=np.float32),
            'metrics': metrics,
            'lag_samples': int(meta[3]),
            'lag_seconds': float(meta[4]),
            'xcorr_score': float(meta[5]) if len(meta) > 5 else float('nan'),
            'lag_clipped': bool(meta[6]) if len(meta) > 6 else False,
            'sync_method': sync_method,
            'joint': joint,
        }
        trial_data[key] = entry
    return trial_data


def load_full_cache(path: Path = FULL_CACHE) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f'Missing full cache: {path}')
    data = np.load(str(path), allow_pickle=True)
    channels = [str(c) for c in data['channels']]
    display_names = [FULL_DISPLAY_NAMES.get(c, c) for c in channels]
    trial_data = _load_trial_map(data, channels, joint=None)
    print(f'[3-DoF] Loaded {len(trial_data)} trials | channels={channels}')
    return {
        'model': '3-DoF',
        'joint': 'all',
        'channels': channels,
        'display_names': display_names,
        'TRIAL_DATA': trial_data,
    }


def load_per_joint_cache(path: Path) -> dict:
    data = np.load(str(path), allow_pickle=True)
    joint = str(data['joint'].item()) if 'joint' in data.files else path.stem.split('_')[-1]
    channels = [str(c) for c in data['channels']]
    display_names = (
        [str(c) for c in data['display_names']]
        if 'display_names' in data.files
        else JOINT_DISPLAY.get(joint, channels)
    )
    trial_data = _load_trial_map(data, channels, joint=joint)
    print(f'[1-DoF/{joint}] Loaded {len(trial_data)} trials | channels={channels}')
    return {
        'model': '1-DoF',
        'joint': joint,
        'channels': channels,
        'display_names': display_names,
        'TRIAL_DATA': trial_data,
    }


BUNDLES: Dict[str, dict] = {}
if FULL_CACHE.is_file():
    BUNDLES['3-DoF'] = load_full_cache()
for joint, path in JOINT_CACHE_PATHS.items():
    if path.is_file():
        BUNDLES[f'1-DoF/{joint}'] = load_per_joint_cache(path)

if not BUNDLES:
    raise FileNotFoundError('No Awinda caches found — run process notebooks first.')
print('Bundles:', list(BUNDLES.keys()))


[3-DoF] Loaded 40 trials | channels=['hip_flexion_r', 'knee_angle_r', 'ankle_angle_r', 'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l']
[1-DoF/hip] Loaded 40 trials | channels=['hip_flexion_r', 'hip_flexion_l']
[1-DoF/knee] Loaded 40 trials | channels=['knee_angle_r', 'knee_angle_l']
[1-DoF/ankle] Loaded 40 trials | channels=['ankle_angle_r', 'ankle_angle_l']
Bundles: ['3-DoF', '1-DoF/hip', '1-DoF/knee', '1-DoF/ankle']


## 1. Flag outliers

A **channel-row** is flagged if `RMSE > 0.25` **or** `R² < 0.6`. Tables below pool all loaded models.


In [3]:
def build_metrics_table(bundles: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for bundle_key, bundle in bundles.items():
        for trial, d in bundle['TRIAL_DATA'].items():
            for m in d['metrics']:
                ch = m['channel']
                if bundle['joint'] == 'all':
                    if 'hip' in ch:
                        joint = 'hip'
                    elif 'knee' in ch:
                        joint = 'knee'
                    else:
                        joint = 'ankle'
                else:
                    joint = bundle['joint']
                rows.append({
                    'bundle': bundle_key,
                    'model': bundle['model'],
                    'joint': joint,
                    'trial': trial,
                    'subject': d['subject'],
                    'condition': d['condition'],
                    'channel': ch,
                    'display': FULL_DISPLAY_NAMES.get(ch, ch),
                    'rmse': m['rmse_nmpkg'],
                    'r2': m['r2_nmpkg'],
                    'lag_s': d['lag_seconds'],
                    'xcorr': d['xcorr_score'],
                })
    df = pd.DataFrame(rows)
    df['fail_rmse'] = df['rmse'] > MAX_RMSE
    df['fail_r2'] = df['r2'] < MIN_R2
    df['flagged'] = df['fail_rmse'] | df['fail_r2']
    df['fail_reason'] = df.apply(
        lambda r: ', '.join(
            ([f'RMSE={r.rmse:.3f}'] if r.fail_rmse else [])
            + ([f'R²={r.r2:.3f}'] if r.fail_r2 else [])
        ) or '',
        axis=1,
    )
    return df.sort_values(['flagged', 'model', 'joint', 'r2'], ascending=[False, True, True, True]).reset_index(drop=True)


metrics = build_metrics_table(BUNDLES)
flagged = metrics[metrics['flagged']].copy()

print(
    f'Flagged channel-rows: {len(flagged)} / {len(metrics)} '
    f'(RMSE > {MAX_RMSE} or R² < {MIN_R2})'
)
print('\nBy model:')
display(
    metrics.groupby('model')
    .agg(n=('channel', 'size'), n_flagged=('flagged', 'sum'), frac=('flagged', 'mean'))
    .assign(frac=lambda x: (100 * x['frac']).round(1))
    .rename(columns={'frac': 'flagged_%'})
)
print('\nBy model × joint:')
display(
    metrics.groupby(['model', 'joint'])
    .agg(n=('channel', 'size'), n_flagged=('flagged', 'sum'), frac=('flagged', 'mean'))
    .assign(frac=lambda x: (100 * x['frac']).round(1))
    .rename(columns={'frac': 'flagged_%'})
)

print('\n=== Flagged channel-rows ===')
display(
    flagged[
        ['model', 'joint', 'trial', 'channel', 'rmse', 'r2', 'fail_reason', 'lag_s', 'xcorr']
    ].reset_index(drop=True).round(4)
)

# Trial-level: any channel flagged within that bundle
trial_flags = (
    metrics.groupby(['bundle', 'model', 'joint', 'trial', 'subject', 'condition'], as_index=False)
    .agg(
        n_ch=('channel', 'size'),
        n_flagged=('flagged', 'sum'),
        mean_rmse=('rmse', 'mean'),
        mean_r2=('r2', 'mean'),
        min_r2=('r2', 'min'),
        max_rmse=('rmse', 'max'),
    )
)
trial_flags['flagged'] = trial_flags['n_flagged'] > 0
print('\n=== Trials with ≥1 flagged channel ===')
display(
    trial_flags[trial_flags['flagged']]
    .sort_values(['model', 'joint', 'min_r2'])
    .reset_index(drop=True)
    .round(4)
)

out_csv = CACHE_DIR / 'check_awinda_outliers_channels.csv'
flagged.to_csv(out_csv, index=False)
trial_csv = CACHE_DIR / 'check_awinda_outliers_trials.csv'
trial_flags[trial_flags['flagged']].to_csv(trial_csv, index=False)
print(f'Saved → {out_csv.name}, {trial_csv.name}')


Flagged channel-rows: 91 / 480 (RMSE > 0.25 or R² < 0.6)

By model:


,n,n_flagged,flagged_%
model,,,
1-DoF,240,61,25.4
3-DoF,240,30,12.5



By model × joint:


n  n_flagged  flagged_%
model joint                          
1-DoF ankle  80         18       22.5
      hip    80         20       25.0
      knee   80         23       28.7
3-DoF ankle  80          7        8.8
      hip    80         11       13.8
      knee   80         12       15.0


=== Flagged channel-rows ===


,model,joint,trial,channel,rmse,r2,fail_reason,lag_s,xcorr
0,1-DoF,ankle,AB05_Maria::LG_0p8mps,ankle_angle_r,0.7566,0.0381,"RMSE=0.757, R²=0.038",1.96,2.8493
1,1-DoF,ankle,AB06_Jimin::RD_0p8mps,ankle_angle_r,0.3208,0.4816,"RMSE=0.321, R²=0.482",8.42,3.9328
2,1-DoF,ankle,AB07_Amy::LG_1p6mps,ankle_angle_r,0.4469,0.4826,"RMSE=0.447, R²=0.483",2.39,3.9666
3,1-DoF,ankle,AB08_Seokhyun::RD_0p8mps,ankle_angle_r,0.3809,0.5124,"RMSE=0.381, R²=0.512",1.83,3.9561
4,1-DoF,ankle,AB06_Jimin::RD_0p8mps,ankle_angle_l,0.3031,0.5190,"RMSE=0.303, R²=0.519",8.42,3.9328
...,...,...,...,...,...,...,...,...,...
86,3-DoF,knee,AB01_Jinwoo::RD_0p8mps,knee_angle_r,0.2630,0.4805,"RMSE=0.263, R²=0.481",7.46,3.9184
87,3-DoF,knee,AB07_Amy::LG_0p8mps,knee_angle_r,0.1039,0.5955,R²=0.596,11.78,3.9498
88,3-DoF,knee,AB02_Oscar::RA_0p8mps,knee_angle_l,0.2545,0.6733,RMSE=0.254,2.97,3.9759
89,3-DoF,knee,AB01_Jinwoo::LG_1p6mps,knee_angle_l,0.2843,0.7546,RMSE=0.284,13.00,3.9379



=== Trials with ≥1 flagged channel ===


,bundle,model,joint,trial,subject,condition,n_ch,n_flagged,mean_rmse,mean_r2,min_r2,max_rmse,flagged
0,1-DoF/ankle,1-DoF,ankle,AB05_Maria::LG_0p8mps,AB05_Maria,LG_0p8mps,2,2,0.5782,0.2851,0.0381,0.7566,True
1,1-DoF/ankle,1-DoF,ankle,AB06_Jimin::RD_0p8mps,AB06_Jimin,RD_0p8mps,2,2,0.3120,0.5003,0.4816,0.3208,True
2,1-DoF/ankle,1-DoF,ankle,AB07_Amy::LG_1p6mps,AB07_Amy,LG_1p6mps,2,2,0.3630,0.6334,0.4826,0.4469,True
3,1-DoF/ankle,1-DoF,ankle,AB08_Seokhyun::RD_0p8mps,AB08_Seokhyun,RD_0p8mps,2,2,0.3530,0.5642,0.5124,0.3809,True
4,1-DoF/ankle,1-DoF,ankle,AB08_Seokhyun::RA_0p8mps,AB08_Seokhyun,RA_0p8mps,2,2,0.3270,0.6653,0.6148,0.3637,True
5,1-DoF/ankle,1-DoF,ankle,AB01_Jinwoo::RA_0p8mps,AB01_Jinwoo,RA_0p8mps,2,1,0.2970,0.7843,0.7067,0.3558,True
6,1-DoF/ankle,1-DoF,ankle,AB01_Jinwoo::LG_0p8mps,AB01_Jinwoo,LG_0p8mps,2,1,0.2463,0.8553,0.8454,0.2747,True
7,1-DoF/ankle,1-DoF,ankle,AB01_Jinwoo::LG_1p2mps,AB01_Jinwoo,LG_1p2mps,2,2,0.2889,0.8545,0.8501,0.2900,True
8,1-DoF/ankle,1-DoF,ankle,AB01_Jinwoo::LG_1p6mps,AB01_Jinwoo,LG_1p6mps,2,2,0.2599,0.9235,0.9164,0.2620,True
9,1-DoF/ankle,1-DoF,ankle,AB08_Seokhyun::LG_1p6mps,AB08_Seokhyun,LG_1p6mps,2,2,0.2917,0.9567,0.9542,0.3277,True


Saved → check_awinda_outliers_channels.csv, check_awinda_outliers_trials.csv


## 2. Interactive moment viewer

Pick **model bundle**, optionally restrict to **flagged-only**, then browse trials. Plots OpenSim ID (GT) vs model prediction.


In [ ]:
bundle_keys = list(BUNDLES.keys())
bundle_dd = widgets.Dropdown(options=bundle_keys, value=bundle_keys[0], description='Model:')
filter_dd = widgets.Dropdown(
    options=[
        ('Flagged only (RMSE>0.25 or R²<0.6)', 'flagged'),
        ('All trials', 'all'),
    ],
    value='flagged',
    description='Show:',
    layout=widgets.Layout(width='340px'),
)
trial_dd = widgets.Dropdown(description='Trial:', layout=widgets.Layout(width='420px'))
unit_dd = widgets.Dropdown(options=['N·m/kg', 'N·m'], value='N·m/kg', description='Unit:')
time_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, layout=widgets.Layout(width='700px'),
)
status_html = widgets.HTML()
out = widgets.Output()


def _bundle():
    return BUNDLES[bundle_dd.value]


def _flagged_trials_for_bundle(bundle_key: str) -> set:
    return set(flagged.loc[flagged['bundle'] == bundle_key, 'trial'].unique())


def _trial_label(bundle_key: str, trial_key: str) -> str:
    sub = flagged[(flagged['bundle'] == bundle_key) & (flagged['trial'] == trial_key)]
    if sub.empty:
        return trial_key
    bits = []
    for _, r in sub.iterrows():
        bits.append(f"{r['display']}:{r['fail_reason']}")
    return f"{trial_key}  [{'; '.join(bits)}]"


def _trial_time_rel(trial_key: str) -> np.ndarray:
    t = np.asarray(_bundle()['TRIAL_DATA'][trial_key]['t'], dtype=np.float64)
    return t - t[0]


def _set_slider(trial_key: str) -> None:
    t_rel = _trial_time_rel(trial_key)
    tmin, tmax = float(t_rel[0]), float(t_rel[-1])
    step = max((tmax - tmin) / 500, 1e-3)
    with time_slider.hold_sync():
        time_slider.min = tmin
        time_slider.max = tmax
        time_slider.step = step
        time_slider.value = (tmin, tmax)


def _refresh_trials(*_) -> None:
    bundle_key = bundle_dd.value
    bundle = _bundle()
    flagged_set = _flagged_trials_for_bundle(bundle_key)
    if filter_dd.value == 'flagged':
        keys = sorted(flagged_set & set(bundle['TRIAL_DATA']))
        if not keys:
            trial_dd.options = [('(no flagged trials)', '')]
            trial_dd.value = ''
            status_html.value = f'<b>{bundle_key}</b>: no flagged trials under current thresholds.'
            return
    else:
        keys = sorted(bundle['TRIAL_DATA'])
    opts = [(_trial_label(bundle_key, k), k) for k in keys]
    # Prefer keeping selection if still valid
    prev = trial_dd.value
    trial_dd.options = opts
    values = [v for _, v in opts]
    trial_dd.value = prev if prev in values else values[0]
    n_flag = len(flagged_set)
    status_html.value = (
        f'<b>{bundle_key}</b>: showing {len(keys)} / {len(bundle["TRIAL_DATA"])} trials '
        f'({n_flag} flagged channel-trials in this bundle).'
    )


def _draw_id(trial_key: str, unit: str, t_window) -> None:
    bundle = _bundle()
    d = bundle['TRIAL_DATA'][trial_key]
    names = bundle['display_names']
    channels = bundle['channels']
    n_ch = len(names)
    t = _trial_time_rel(trial_key)
    scale = 1.0 if unit == 'N·m/kg' else float(d['mass_kg'])
    y_pred = np.asarray(d['pred_nmpkg'], dtype=np.float64) * scale
    y_gt = np.asarray(d['id_nmpkg'], dtype=np.float64) * scale
    n = min(len(t), len(y_pred), len(y_gt))
    t, y_pred, y_gt = t[:n], y_pred[:n], y_gt[:n]
    t0, t1 = t_window

    # Layout: 3-DoF → 3x2; 1-DoF → 1xn
    if n_ch == 6:
        fig, axs = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
        axs = axs.ravel()
    else:
        fig, axs = plt.subplots(1, n_ch, figsize=(6.5 * n_ch, 4), sharey=True)
        if n_ch == 1:
            axs = [axs]

    for c, ax in enumerate(axs):
        m = d['metrics'][c]
        is_bad = (m['rmse_nmpkg'] > MAX_RMSE) or (m['r2_nmpkg'] < MIN_R2)
        ax.plot(t, y_gt[:, c], label='OpenSim ID (GT)', color='#1e88e5', lw=1.8)
        ax.plot(t, y_pred[:, c], label='Model', color='#e53935', lw=1.4, ls='--')
        badge = ' ⚠' if is_bad else ''
        ax.set_title(
            f"{names[c]}{badge} | RMSE={m['rmse_nmpkg'] * scale:.3f}, R²={m['r2_nmpkg']:.3f}",
            color='#b71c1c' if is_bad else 'black',
        )
        ax.set_xlim(t0, t1)
        ax.set_ylabel(unit)
        ax.set_xlabel('Time (s)')
        ax.axhline(0, color='gray', lw=0.6, ls=':')
        if c == 0:
            ax.legend(loc='best', fontsize=9)

    fig.suptitle(
        f"{bundle_dd.value} | {trial_key} | lag={d['lag_samples']:+d} "
        f"({d['lag_seconds']:+.3f} s) | xcorr={d['xcorr_score']:.3f}",
        fontsize=12,
    )
    fig.tight_layout()
    with out:
        clear_output(wait=True)
        plt.show()


def _redraw(*_) -> None:
    trial_key = trial_dd.value
    if not trial_key:
        with out:
            clear_output(wait=True)
            print('No trial selected.')
        return
    _draw_id(trial_key, unit_dd.value, time_slider.value)


def _on_bundle_or_filter(change=None) -> None:
    _refresh_trials()
    if trial_dd.value:
        _set_slider(trial_dd.value)
    _redraw()


def _on_trial(change) -> None:
    if not change['new']:
        return
    _set_slider(change['new'])
    _redraw()


bundle_dd.observe(_on_bundle_or_filter, names='value')
filter_dd.observe(_on_bundle_or_filter, names='value')
trial_dd.observe(_on_trial, names='value')
unit_dd.observe(_redraw, names='value')
time_slider.observe(_redraw, names='value')

_on_bundle_or_filter()
display(widgets.VBox([
    widgets.HBox([bundle_dd, filter_dd, unit_dd]),
    trial_dd,
    status_html,
    time_slider,
    widgets.HTML('<b>Synced ID: OpenSim GT vs model</b>'),
    out,
]))
